# Quality Control Metrics for dMRI

You cannot trust results you have not quality-controlled. This notebook shows you **how to systematically check your data** at every stage of the pipeline, using quantitative metrics rather than visual inspection alone.

## Why QC matters

- A dataset with 30% motion-corrupted volumes will give plausible-looking FA maps — but wrong ones.
- A failed eddy correction silently shifts the bvec frame → all downstream models are fitting the wrong directions.
- Gibbs ringing, if not removed, artificially increases FA near CSF boundaries.

**Silent failures are the enemy.** This module teaches you to make them loud.

---

In [ ]:
import sys, subprocess
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

sys.path.insert(0, '../../scripts')

data_dir = Path('../../data/hcp/100307/T1w/Diffusion')
prep_dir = Path('../../data/hcp/100307/preprocessed')
dti_dir  = Path('../../data/hcp/100307/dti')
qc_dir   = Path('../../data/hcp/100307/qc')
qc_dir.mkdir(parents=True, exist_ok=True)

dwi_raw    = str(data_dir / 'data.nii.gz')
bvals_f    = str(data_dir / 'bvals')
bvecs_f    = str(data_dir / 'bvecs')
mask       = str(data_dir / 'nodif_brain_mask.nii.gz')

bvals = np.loadtxt(bvals_f)
bvecs = np.loadtxt(bvecs_f)

img   = nib.load(dwi_raw)
data  = img.get_fdata()
maskd = nib.load(mask).get_fdata().astype(bool)

print(f'Data shape: {data.shape}')
print(f'Shells    : {np.unique(np.round(bvals, -2)).astype(int)}')

## 1. Raw data QC: Signal-to-Noise Ratio per volume

In [ ]:
# SNR for each volume
# Estimate noise from background (outside brain mask)

snr_per_vol = []
for v in range(data.shape[3]):
    brain_signal = data[..., v][maskd].mean()
    bg_noise     = data[..., v][~maskd].std() + 1e-10
    snr_per_vol.append(brain_signal / bg_noise)

snr_per_vol = np.array(snr_per_vol)

# Flag outlier volumes (SNR < mean - 2*std)
snr_mean = snr_per_vol.mean()
snr_std  = snr_per_vol.std()
low_snr_vols = np.where(snr_per_vol < snr_mean - 2 * snr_std)[0]

fig, ax = plt.subplots(figsize=(14, 4))
colors = ['red' if v in low_snr_vols else
          ('navy' if bvals[v] < 50 else
           ('royalblue' if bvals[v] < 1100 else
            ('tomato' if bvals[v] < 2100 else 'darkred')))
          for v in range(len(snr_per_vol))]

ax.bar(range(len(snr_per_vol)), snr_per_vol, color=colors, edgecolor='none')
ax.axhline(snr_mean - 2 * snr_std, color='red', linestyle='--', label='Mean - 2σ (outlier threshold)')
ax.set_xlabel('Volume index')
ax.set_ylabel('SNR (signal / background noise)')
ax.set_title('Per-volume SNR  |  Red bars = outliers  |  Colour by shell (b=0/1000/2000/3000)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Mean SNR: {snr_mean:.1f} ± {snr_std:.1f}')
print(f'Outlier volumes: {list(low_snr_vols)}')
print(f'Expected: b=0 > b=1000 > b=2000 > b=3000 (higher b = lower signal = lower SNR)')

## 2. Eddy current QC: EDDY QUAD

In [ ]:
# ─── [FSL] eddy_quad ─────────────────────────────────────────────────────────
#
# eddy_quad generates a comprehensive QC report for FSL eddy output.
# It produces an HTML report + summary statistics.

eddy_base = str(prep_dir / 'dwi_eddy')
quad_dir  = str(qc_dir / 'eddy_quad')

if Path(eddy_base + '.eddy_parameters').exists():
    quad_cmd = [
        'eddy_quad',
        eddy_base,
        '-idx',  str(prep_dir / 'index.txt'),
        '-par',  str(prep_dir / 'acqparams.txt'),
        '-m',    mask,
        '-b',    bvals_f,
        '-g',    bvecs_f,
        '-o',    quad_dir,
        '-f',
    ]
    print('[FSL] eddy_quad command:')
    print(' '.join(quad_cmd))
    result = subprocess.run(quad_cmd, capture_output=True, text=True)
    print(result.stdout or '(done)')
    print(f'\nOpen {quad_dir}/qc.pdf for the full report')
else:
    print('eddy output not found — run FSL eddy first')
    print()
    print('eddy_quad generates:')
    print('  • qc.pdf — visual QC report')
    print('  • qc.json — machine-readable metrics')
    print('  • Average_SNR_per_b_across_shells.png')
    print('  • Absolute_motion_stats.png')
    print('  • Outlier_slice_stats.png')

In [ ]:
# Parse eddy QC JSON if available
import json

qc_json = Path(quad_dir) / 'qc.json'
if qc_json.exists():
    with open(qc_json) as f:
        qc = json.load(f)

    print('=== eddy_quad summary ===')
    print(f'  Absolute motion (mean)   : {qc["qc_mot_abs"]:.3f} mm')
    print(f'  Relative motion (mean)   : {qc["qc_mot_rel"]:.3f} mm')
    print(f'  Outlier slices (%)       : {qc["qc_outliers_pe"]:.1f}%')

    # Reference values (Maximov et al. 2019 large cohort)
    print()
    print('Reference ranges (healthy adults, 3T):')
    print('  Absolute motion < 1.0 mm = good; > 2.0 mm = re-scan if possible')
    print('  Outlier slices < 5%      = good; > 20%     = exclude from study')

## 3. DTI model QC: FA and MD distributions

In [ ]:
# Load DTI metrics
fa_path = str(dti_dir / 'fsl_dti_FA.nii.gz') if (dti_dir / 'fsl_dti_FA.nii.gz').exists() \
          else str(dti_dir / 'dipy_FA.nii.gz')
md_path = str(dti_dir / 'fsl_dti_MD.nii.gz') if (dti_dir / 'fsl_dti_MD.nii.gz').exists() \
          else str(dti_dir / 'dipy_MD.nii.gz')

if Path(fa_path).exists():
    FA = nib.load(fa_path).get_fdata()
    MD = nib.load(md_path).get_fdata()

    # Separate WM voxels (FA > 0.2)
    wm_mask = (FA > 0.2) & maskd

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(FA[wm_mask], bins=50, color='steelblue', edgecolor='none', density=True)
    axes[0].axvline(FA[wm_mask].mean(), color='red', label=f'Mean = {FA[wm_mask].mean():.2f}')
    axes[0].set_title('FA distribution (WM mask, FA > 0.2)')
    axes[0].set_xlabel('FA')
    axes[0].legend()

    axes[1].hist(MD[wm_mask] * 1e3, bins=50, color='tomato', edgecolor='none', density=True)
    axes[1].axvline(MD[wm_mask].mean() * 1e3, color='navy',
                    label=f'Mean = {MD[wm_mask].mean()*1e3:.2f} ×10⁻³ mm²/s')
    axes[1].set_title('MD distribution (WM mask)  [×10⁻³ mm²/s]')
    axes[1].set_xlabel('MD ×10⁻³ mm²/s')
    axes[1].legend()

    plt.suptitle('DTI metric distributions — sanity check', fontsize=12)
    plt.tight_layout()
    plt.show()

    print('Expected ranges (healthy adult WM, 3T, b=1000):')
    print('  FA mean  : 0.40 – 0.55')
    print('  MD mean  : 0.70 – 0.85 ×10⁻³ mm²/s')
    print()
    print(f'This subject: FA = {FA[wm_mask].mean():.3f}, MD = {MD[wm_mask].mean()*1e3:.3f}')
else:
    print('DTI metrics not found — run Module 2 first')

## 4. CSD QC: Response function and FOD voxels

In [ ]:
# ─── [MRtrix3] mrinfo to check FOD stats ──────────────────────────────────────
csd_dir = Path('../../data/hcp/100307/csd')
wm_fod  = str(csd_dir / 'wmfod_norm.mif')

if Path(wm_fod).exists():
    result = subprocess.run(['mrinfo', wm_fod, '-size', '-datatype'],
                            capture_output=True, text=True)
    print('[MRtrix3] FOD info:')
    print(result.stdout)

    # Check FOD amplitudes are not negative (CSD constraint check)
    result2 = subprocess.run([
        'mrmath', wm_fod, 'min', '-axis', '3',
        '-', '-force', '|', 'mrstats', '-',
    ], capture_output=True, text=True, shell=False)
    # Alternative: convert and check in numpy
    print('FOD amplitude check via mrstats:')
    print('  $ mrstats wmfod_norm.mif    (shows min/max/mean of all SH coefficients)')
else:
    print('Run Module 2 (CSD) first to generate FODs.')

# Response function QC: plot
resp_files = {
    'WM':  str(csd_dir / 'response_wm.txt'),
    'GM':  str(csd_dir / 'response_gm.txt'),
    'CSF': str(csd_dir / 'response_csf.txt'),
}

fig, ax = plt.subplots(figsize=(7, 4))
shells_available = []

for label, path in resp_files.items():
    if Path(path).exists():
        resp = np.loadtxt(path)
        if resp.ndim == 1:
            resp = resp[np.newaxis, :]
        # First column = b-value, rest = SH coefficients
        for row in resp:
            ax.plot(row[1:], label=f'{label} (b={int(row[0])})', marker='o', markersize=4)

if Path(list(resp_files.values())[0]).exists():
    ax.axhline(0, color='grey', linestyle='--', alpha=0.5)
    ax.set_xlabel('SH coefficient index')
    ax.set_ylabel('Amplitude')
    ax.set_title('Response functions per tissue (dhollander)')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    print('Expected:')
    print('  WM  : large l=0 coefficient, decreasing with l')
    print('  GM  : only l=0 (isotropic)')
    print('  CSF : only l=0, largest amplitude (highest diffusivity)')

## 5. Tractography QC: Streamline length distribution

In [ ]:
# ─── [MRtrix3] tckstats ───────────────────────────────────────────────────────
tck_dir = Path('../../data/hcp/100307/tractography')
tck_file = str(tck_dir / 'prob_iFOD2_1M.tck')

if Path(tck_file).exists():
    result = subprocess.run([
        'tckstats', tck_file,
        '-histogram', str(qc_dir / 'streamline_lengths.csv'),
        '-force'
    ], capture_output=True, text=True)
    print('[MRtrix3] tckstats output:')
    print(result.stdout)

    hist_file = qc_dir / 'streamline_lengths.csv'
    if hist_file.exists():
        hist = np.loadtxt(hist_file, delimiter=',')
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar(hist[:, 0], hist[:, 1], width=hist[1, 0] - hist[0, 0],
               color='steelblue', edgecolor='none')
        ax.set_xlabel('Streamline length (mm)')
        ax.set_ylabel('Count')
        ax.set_title('Streamline length distribution')
        plt.tight_layout()
        plt.show()
        print('Expected: peak around 40-80 mm; long tail up to ~250 mm')
        print('Red flags: very many very short (<10 mm) or very long (>250 mm) streamlines')
else:
    print('Run tractography (Module 3) first.')

# ─── [DIPY] Streamline length analysis ────────────────────────────────────────
from dipy.io.streamline import load_trk

dipy_trk = str(tck_dir / 'prob_dipy.trk')
if Path(dipy_trk).exists():
    sft = load_trk(dipy_trk, nib.load(dwi_raw))
    sls = sft.streamlines
    lengths_mm = np.array([len(s) * 0.5 for s in sls])

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(lengths_mm, bins=50, color='tomato', edgecolor='none', density=True)
    ax.set_xlabel('Streamline length (mm)')
    ax.set_ylabel('Density')
    ax.set_title(f'DIPY streamline length distribution ({len(sls):,} streamlines)')
    plt.tight_layout()
    plt.show()

## 6. QC summary report

In [ ]:
# Build a structured QC table

qc_results = {}

# Raw data
b0_snr = np.mean([snr_per_vol[i] for i in range(len(bvals)) if bvals[i] < 50])
qc_results['b=0 SNR (mean)']   = f'{b0_snr:.1f}'
qc_results['Outlier volumes']  = f'{len(low_snr_vols)} / {len(bvals)}'

# DTI
if Path(fa_path).exists():
    FA_wm = nib.load(fa_path).get_fdata()[wm_mask]
    qc_results['FA mean (WM)'] = f'{FA_wm.mean():.3f}'
    qc_results['FA std  (WM)'] = f'{FA_wm.std():.3f}'

# Tractography
if Path(dipy_trk).exists():
    qc_results['Streamlines (DIPY)'] = f'{len(sls):,}'
    qc_results['Mean length (mm)']   = f'{lengths_mm.mean():.0f}'

print('=== QC SUMMARY — Subject 100307 ===')
print(f'  Date processed : {pd.Timestamp.now().strftime("%Y-%m-%d")}')
print()
for k, v in qc_results.items():
    flag = ''
    if 'SNR' in k and float(v) < 10:
        flag = '  ⚠️  LOW'
    elif 'Outlier' in k and int(v.split('/')[0]) > len(bvals) * 0.05:
        flag = '  ⚠️  HIGH'
    print(f'  {k:<30} : {v}{flag}')

# Save to CSV
pd.DataFrame(list(qc_results.items()), columns=['Metric', 'Value']).to_csv(
    str(qc_dir / 'qc_summary.csv'), index=False)
print(f'\nSaved to: {qc_dir / "qc_summary.csv"}')

---

## QC checklist

Print this and use it for every dataset:

- [ ] **Raw data**: Check b=0 SNR (>15 is good; <10 is problematic)
- [ ] **Raw data**: No systematic signal dropout in any shell
- [ ] **Eddy**: Motion < 1 mm mean absolute displacement
- [ ] **Eddy**: Outlier slices < 5% of total
- [ ] **Eddy**: Rotated bvecs used in all downstream steps
- [ ] **Denoising**: Noise map is spatially smooth (no anatomical structure)
- [ ] **DTI**: FA range 0–1; MD ≈ 0.7–2.5×10⁻³ mm²/s in WM
- [ ] **CSD**: WM response function has correct shape (decreasing SH order)
- [ ] **Tractography**: Length distribution has expected shape
- [ ] **Tractography**: No implausibly long streamlines (> 300 mm)

**Next**: [Critically evaluating your results →](02_critical_evaluation.ipynb)